# Gemini Research Credits — Setup Guide

Use this notebook as a concise checklist to run your code with Gemini on Google Cloud.

> **Credit restriction:** use the award only for **Gemini models** and other explicitly approved Google Cloud SKUs. Do **not** use third-party models (e.g. Anthropic or DeepSeek) through Model Garden / Marketplace unless separately approved.

---
## **1. To use the Gemini API (Standard Inference) - [Student]**

Install the Google Gen AI SDK, then enter the API key securely when prompted.

In [ ]:
%pip install -q google-genai

In [ ]:
"""
Option 1: If you are running locally (e.g., a Slurm cluster like Hábrók or Snellius)
"""

import os
from getpass import getpass

# Skippable if you already configured the key in your environment
os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key: ")
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

In [ ]:
"""
Option 2: If you are running in Colab (e.g., a Jupyter Notebook)
"""

from google.colab import userdata

# Add the key to Colab Secrets by clicking the 🔑 icon on the left
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
"""
Running
"""

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-3.8-flash"
config = types.GenerateContentConfig(
    temperature=1.0,
    top_p=0.95,
    top_k=40,
    max_output_tokens=500,
    thinking_config=types.ThinkingConfig(
        thinking_level="low", # Couldn't be shut down. Default:  medium (i.e., None). Choices: low, medium, high
        include_thoughts=True, # To enable the thinking summary in the response  (Note: we couldn't get the raw CoT chain for Gemini models)
    )
)

response = client.models.generate_content(
    model=MODEL,
    contents="Explain photosynthesis in one paragraph.",
    config=config,
)

In [ ]:
print("Response answer:")
print(response.text, '\n')
print("Thinking chain:")
for part in response.candidates[0].content.parts:
  if part.thought:
    print(part.text)

See all available Gemini models: https://ai.google.dev/gemini-api/docs/models

Find the prices: https://ai.google.dev/gemini-api/docs/pricing

> **🚨🚨🚨 Important: ALWAYS** run a smoke test with a small subset before full-corpus running to **estimate the total price!**

---
## **2. [Recommended] Lower-cost Batch Inference - [Student]**

- **Batch API:** asynchronous bulk processing at **50% of standard cost**; best when results are not needed immediately (target turnaround up to 24 hours).
- It **saves a large amount of the budgets💰** assigned to your account.
- Below is an example to submit two prompts as one asynchronous batch job, waits for completion, and print the results.

In [ ]:
"""
Submit your job in a jsonl structure. It will run when there are available resources.
After the job is done, you can pull the results back to your local machine.
"""
import time

requests = [
    {"contents": [{"role": "user", "parts": [{"text": "Explain photosynthesis in one sentence."}]}],
     "config": {
            "temperature": 1.0, "top_p": 0.95, "top_k": 40, "max_output_tokens": 500,
            "thinking_config": {
                "thinking_level": "low",
                "include_thoughts": True,
            },
      },},
    {"contents": [{"role": "user", "parts": [{"text": "Explain gravity in one sentence."}]}],
     "config": {
            "temperature": 1.0, "top_p": 0.95, "top_k": 40, "max_output_tokens": 500,
            "thinking_config": {
                "thinking_level": "low",
                "include_thoughts": True,
            },
      },},
]

# Submission
batch = client.batches.create(
    model="gemini-3.8-flash",
    src=requests,
    config={"display_name": "demo-batch"},
)

done = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}
print(f"Job name: {batch.name}. This will be necessary to get the results later on.")

In [ ]:
# Check the status of existing batch jobs
for job in client.batches.list():
    print(
        job.name,
        "|", job.display_name,
        "|", job.state.name
    )

batches/8p263y5xajt281u9khzzz2lqinwh8013hovr | demo-batch | JOB_STATE_SUCCEEDED
batches/sljw4mqy9n87x6ak145w5058uvn7vptz6a28 | demo-batch | JOB_STATE_SUCCEEDED


In [ ]:
# If you like, you may run a loop to wait for it
while True:
    batch = client.batches.get(name=batch.name)
    if batch.state.name in done:
        break
    time.sleep(10)

# Retrieve the results with JOB NAME once the batch inference is done
print("Final state:", batch.state.name, "\n")
if batch.state.name == "JOB_STATE_SUCCEEDED":
    for i, r in enumerate(batch.dest.inlined_responses, 1):
        response = r.response

        thoughts = []
        answers = []

        for part in response.candidates[0].content.parts:
            if getattr(part, "thought", False):
                thoughts.append(part.text)
            elif part.text:
                answers.append(part.text)

        print(f"\nRequest {i}")
        print("Thought summary:", "\n".join(thoughts) or "None")
        print("Answer:", "\n".join(answers))
        print("Thinking tokens:", response.usage_metadata.thoughts_token_count)

---
## **3. After running - [Student]**

- Check **Billing → Reports / Credits** to see the cost and to confirm usage is being offset by the Gemini credit.